converting between geographic coordinates (WGS84, lat/lon) and a local Cartesian coordinate system via UTM32650 whos original point - ws point

In [26]:
from pyproj import Proj, transform
import geopy.distance


# intersection: 116.483738,39.995377 East exit of LSH: 116.482958,39.99682
# test points ws:116.45911,39.73473 嘉宝花苑 ne:116.48428,39.750821 金茂艺墅
# def wgs84_to_utm(points=[(39.70152574, 116.43420538),(39.75155747, 116.52821347)]):
def wgs84_to_utm(points=[(39.73469819, 116.45506495),(39.74512695, 116.46600579)]):

    # 定义西南点和东北点的经纬度
    # sw_lat, sw_lon = 39.70152574, 116.43420538
    # ne_lat, ne_lon = 39.75155747, 116.52821347
    
    # test points ws:116.45911,39.73473 嘉宝花苑 ne:116.48428,39.750821 金茂艺墅
    # sw_lat, sw_lon = 39.73473, 116.45911
    # ne_lat, ne_lon = 39.750821, 116.48428
    #[(0.0, 0.0), (2167.0887929027667, 1773.1645824313164)] , 误差接近20m

    #mzone
    sw_lat, sw_lon = 39.73469819, 116.45506495
    ne_lat, ne_lon = 39.74512695, 116.46600579
    # [(0.0, 0.0), (944.4153076584917, 1151.8048132052645)]

    
    # 定义要转换的点（示例点）
    points 

    # 计算西南点的UTM坐标（使用WGS84）
    proj_wgs84 = Proj(init='epsg:4326', proj='latlong', datum='WGS84')
    proj_utm = Proj( init= 'epsg:32650', proj='utm')  # 北京地区 根据经度选择UTM Zone 50

    sw_utm_x, sw_utm_y = transform(proj_wgs84, proj_utm, sw_lon, sw_lat)

    # 定义函数转换坐标
    def convert_to_local(lat, lon):
        utm_x, utm_y = transform(proj_wgs84, proj_utm, lon, lat)
        local_x = utm_x - sw_utm_x
        local_y = utm_y - sw_utm_y
        return local_x, local_y

    # 转换所有点
    local_points = [convert_to_local(lat, lon) for lat, lon in points]

    # 输出结果
    for i, (local_x, local_y) in enumerate(local_points):
        print(f"Point {i+1}: Local X = {local_x:.2f} m, Local Y = {local_y:.2f} m")
    return local_points
#[(0.0, 0.0), (2167.0887929027667, 1773.1645824313164)]
#Measured：
# North-south distance: 1786.58 km
# East-west distance: 2157.66 km


wgs84_to_utm()

Point 1: Local X = 0.00 m, Local Y = 0.00 m
Point 2: Local X = 944.42 m, Local Y = 1151.80 m


/home/qichen/anaconda3/envs/hdmapnet_py38/lib/python3.8/site-packages/pyproj/crs/crs.py:141: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)
/home/qichen/anaconda3/envs/hdmapnet_py38/lib/python3.8/site-packages/pyproj/crs/crs.py:141: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)
/tmp/ipykernel_1533538/3961890567.py:30: FutureWarning: This function is deprecated. See: https://pyproj4.github.io/pyproj/stable/gotchas.html#upgrading-to-pyproj-2-f

[(0.0, 0.0), (944.4153076584917, 1151.8048132052645)]

wgs84-UTM pyproj

In [24]:
from pyproj import Transformer

points = []

tf = Transformer.from_crs("epsg:4326", "epsg:32650") 
# original points ws:116.45911,39.73473 嘉宝花苑 ne:116.48428,39.750821 金茂艺墅
sw_lat, sw_lon = 39.73473, 116.45911
target_lat, target_lon = 39.750821, 116.48428

# North-south distance: 1.79 km 1920m
# East-west distance: 2.15 km 2049m
lat = 39.73473
lon = 116.45911
# transitted original UTM coords x,y
origx, origy = tf.transform(lat, lon)
print("x:", origx, "y:", origy)

# 定义函数转换坐标
def convert_to_local(target_lat, target_lon, origx,origy):
    utm_x, utm_y = tf.transform(target_lat,target_lon)
    utm_t_x = utm_x - origx
    utm_t_y = utm_y - origy
    return utm_t_x, utm_t_y

utm_t_x, utm_t_y = convert_to_local(target_lat, target_lon, origx,origy)

from pyproj import Proj, Geod
 
# 设置UTM投影，例如对于ZONE BJ
proj = Proj(init='epsg:32649')
 
# 两点坐标 (经度, 纬度)
point1 = (12.456, 41.74)
point2 = (12.567, 41.85)
 
# 将WGS84经纬度转换为UTM坐标
x1, y1 = proj(point1[0], point1[1])
x2, y2 = proj(point2[0], point2[1])
 
# 使用Geod对象计算两点间的大圆距离
geod = Geod(ellps='WGS84')
distance, azimuth, backazimuth = geod.inv(x1, y1, x2, y2)
 
# 输出距离，单位为米
print(f"Distance: {distance:.2f} m")
# # 转换所有点
# local_points = [convert_to_local(lat, lon) for lat, lon in points]

# # 输出结果
# for i, (local_x, local_y) in enumerate(local_points):
#     print(f"Point {i+1}: Local X = {local_x:.2f} m, Local Y = {local_y:.2f} m")

x: 453651.5026364988 y: 4398455.36777962


/home/qichen/anaconda3/envs/hdmapnet_py38/lib/python3.8/site-packages/pyproj/crs/crs.py:141: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)


Distance: nan m


In [27]:
import psycopg2
from psycopg2.extras import RealDictCursor

import psycopg2
from shapely.geometry import shape, Point
from shapely.wkt import dumps
import math
import re
import uuid  


# original coords in gcj02 ws:116.45911,39.73473 嘉宝花苑 ne:116.48428,39.750821 金茂艺墅
# sw_lat, sw_lon = 39.73473, 116.45911
# target_lat, target_lon = 39.750821, 116.48428
# origin_lat, origin_lon = 39.70152574, 116.43420538

#mzone ws , ne
sw_lat, sw_lon = 39.73469819, 116.45506495
ne_lat, ne_lon = 39.74512695, 116.46600579
# Define the points to transform (in WGS84 coordinates)

# Define the earth's radius in meters
R = 6371000

# Define the conversion functions
def lat_to_meter(lat):
    return R * math.radians(lat)

def lon_to_meter(lon, lat):
    return R * math.radians(lon) * math.cos(math.radians(lat))

points_wgs84 = [
    # Add your points here, e.g., [lat1, lon1], [lat2, lon2], ...
]

# # Transform the points to the local coordinate system
# def pts2local(points_wgs84):
#     points_local = []
#     for lat, lon in points_wgs84:
#         dx = lon_to_meter(lon, lat) - lon_to_meter(origin_lon, origin_lat)
#         dy = lat_to_meter(lat) - lat_to_meter(origin_lat)
#         points_local.append([dx, dy])
#     for point in points_local:
#         print(f"Local coordinates: x={point[0]:.2f}m, y={point[1]:.2f}m")
        
# 配置数据库连接参数
conn_params = {
    "dbname": "map",
    "user": "postgres",
    "password": "123",
    "host": "localhost",
    "port": "5432"
}
 
# 连接到数据库
conn = psycopg2.connect(**conn_params)
 
# 创建一个游标对象
cur = conn.cursor(cursor_factory=RealDictCursor)
 
# 执行SQL查询
cur.execute("SELECT ST_AsText(geometry) FROM test_0730_mzone.lane_boundary where oid = 1413 LIMIT 1;")
 
# 获取查询结果
rows = cur.fetchall()

# print(rows)
geometry = rows[0]
# for row in rows:
#     # 假设 'geometry_column' 是你的geometry列名
#     # geometry = row['geometry_column']
#     geometry = row
#     # 操作geometry对象
#     # 处理geometry数据
#     # print(geometry)

# 关闭游标和连接
cur.close()
conn.close()

# Create a Shapely geometry object from the WKT string

# geometry = shapely.wkt.loads(wkt_geometry) in GCJ02!!!
print(geometry)

def wkt2xygeom(geometry):
    # Extract the coordinates from the MULTILINESTRING Z geometry and transit into wgs84 [lat, lon]
    coords = re.findall(r"(\d+(?:\.\d+)?) (\d+(?:\.\d+)?) \d+", geometry['st_astext'])

    # Convert the coordinates to a list of [lat, lon] points
    points_wgs84 = [[float(lat), float(lon)] for lon, lat in coords]
    #in GCJ02!!!
    return points_wgs84
    # Initialize the dictionary

#in GCJ02!!!
xygeoms = wkt2xygeom(geometry)

lane_divider = {
    "token": "03638db7-5d12-42ea-a753-11be36d6e56e",
    "line_token": "a1cca1bb-d8df-433a-b062-a9cf19bb4a69",
    "lane_divider_segments": [
        {
          "node_token": "969b752d-b1ec-451a-91d6-435f1211251e",
          "segment_type": "DOUBLE_DASHED_WHITE"
        }
      ]
}
nodes = []
nodes_token=[]
node = {
      "token": "",
    #   "x": 772.859616346946,
    #   "y": 1867.5701729191703
      "x": 0,
      "y": 0
},
# 为每个点生成node字典  
for point in xygeoms:  
    # 生成随机token  
    token = str(uuid.uuid4())  
    # 创建node字典  
    node = {  
        "token": token,  
        "x": point[1],  # 经度  
        "y": point[0]   # 纬度  
    }  
    # 将node字典添加到列表中  
    nodes.append(node)
    nodes_token.append(node['token'])

# TODO: to be further generated

pass

line_classes=['road_divider(y)' , 'lane_divider(y)'], """y"""
ped_crossing_classes=['ped_crossing(n)'], """No"""
contour_classes=['road_segment(polygon, n)', 'lane(polygon, n)'], """No"""
# Print the resulting dictionary
# print(json.dumps(lane_divider, indent=4))

RealDictRow([('st_astext', 'MULTILINESTRING Z ((116.466070404276 39.7388245537877 0,116.466069314629 39.738828074187 0,116.466062525287 39.738854393363 0,116.466055735946 39.7388809639961 0,116.466048946604 39.7389070317149 0,116.466028578579 39.738986492157 0,116.466024303809 39.7390026692301 0,116.466021537781 39.73901306279 0,116.466014496982 39.739039381966 0,116.466007707641 39.739065952599 0,116.466000918299 39.739092271775 0,116.4659938775 39.7391188424081 0,116.465980298817 39.73917148076 0,116.465973509476 39.739198051393 0,116.465966468677 39.739224370569 0,116.465960014611 39.7392512764782 0,116.465952889994 39.7392775956541 0,116.465946100652 39.7393044177443 0,116.46593939513 39.7393304854631 0,116.46592548117 39.7393836267292 0,116.465918775648 39.7394101973623 0,116.465891366824 39.7395163122565 0,116.465884577483 39.7395428828895 0,116.4658709988 39.7395957726985 0,116.46585716866 39.7396489139646 0,116.465829759836 39.7397544421256 0,116.465822970495 39.7397810127586 0

Calculator for 2 lon/lat distance from direction of south-north and also western and eastern

In [17]:
import math

#Roughly accuracy
def distance_latlon(lat1, lon1, lat2, lon2):
    # Radius of the Earth in kilometers
    R = 6371.0

    # Convert latitudes and longitudes to radians
    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = math.radians(lat2)
    lon2_rad = math.radians(lon2)

    # Calculate the differences in latitude and longitude
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    # Haversine formula
    a = math.sin(dlat/2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

    # Distance in kilometers
    distance = R * c

    return distance

# from math import radians,cos,sin,asin,sqrt,pi,atan,tan,atan2
# 这个算法和 geographiclib 算的几乎一样，相差<0.2米
def distVincenty(lat1, lon1, lat2, lon2):
    '精度更高的椭球计算,计算两个 WGS84 经纬度点的距离'
    a = 6378137.0  # vincentyConstantA(WGS84) ##单位:米
    b = 6356752.3142451  # vincentyConstantB(WGS84) ##单位:米
    f = 1 / 298.257223563  # vincentyConstantF(WGS84)
    L = math.radians(lon2 - lon1)
    U1 = math.atan((1 - f) * math.tan(math.radians(lat1)))
    U2 = math.atan((1 - f) * math.tan(math.radians(lat2)))
    sinU1 = math.sin(U1)
    cosU1 = math.cos(U1)
    sinU2 = math.sin(U2)
    cosU2 = math.cos(U2)
    lambda1 = L
    lambdaP = 2 * math.pi
    iterLimit = 20
 
    sinLambda = 0.0
    cosLambda = 0.0
    sinSigma = 0.0
    cosSigma = 0.0
    sigma = 0.0
    alpha = 0.0
    cosSqAlpha = 0.0
    cos2SigmaM = 0.0
    C = 0.0
    while (abs(lambda1 - lambdaP) > 1e-12 and --iterLimit > 0):
        sinLambda = math.sin(lambda1)
        cosLambda = math.cos(lambda1)
        sinSigma = math.sqrt((cosU2 * sinLambda) * (cosU2 * sinLambda) + (cosU1 * sinU2 - sinU1 * cosU2 * cosLambda) * (
                cosU1 * sinU2 - sinU1 * cosU2 * cosLambda))
        if (sinSigma == 0):
            return 0
        cosSigma = sinU1 * sinU2 + cosU1 * cosU2 * cosLambda
        sigma = math.atan2(sinSigma, cosSigma)
        alpha = math.asin(cosU1 * cosU2 * sinLambda / sinSigma)
        cosSqAlpha = math.cos(alpha) * math.cos(alpha)
        cos2SigmaM = cosSigma - 2 * sinU1 * sinU2 / cosSqAlpha
        C = f / 16 * cosSqAlpha * (4 + f * (4 - 3 * cosSqAlpha))
        lambdaP = lambda1
        lambda1 = L + (1 - C) * f * math.sin(alpha) * (
                sigma + C * sinSigma * (cos2SigmaM + C * cosSigma * (-1 + 2 * cos2SigmaM * cos2SigmaM)))
 
    if iterLimit == 0:
        return 0.0
 
    uSq = cosSqAlpha * (a * a - b * b) / (b * b)
    A = 1 + uSq / 16384 * (4096 + uSq * (-768 + uSq * (320 - 175 * uSq)))
    B = uSq / 1024 * (256 + uSq * (-128 + uSq * (74 - 47 * uSq)))
    deltaSigma = B * sinSigma * (cos2SigmaM + B / 4 * (
            cosSigma * (-1 + 2 * cos2SigmaM * cos2SigmaM) - B / 6 * cos2SigmaM * (-3 + 4 * sinSigma * sinSigma) * (
            -3 + 4 * cos2SigmaM * cos2SigmaM)))
    s = b * A * (sigma - deltaSigma)
    d = s  ##单位:米
    return d

# test from LSH, and test pass!
# intersection: 116.483738,39.995377 East exit of LSH: 116.482958,39.99682
# sw_lat, sw_lon = 39.995377, 116.483738
# ne_lat, ne_lon = 39.99682, 116.482958

#test: 长安街  复兴门 116.356642,39.907317， 天安门西 116.39171,39.907382
# sw_lat, sw_lon = 39.907317, 116.356642
# ne_lat, ne_lon = 39.907382, 116.39171
# North-south distance: 0.01 km
# East-west distance: 2.99 km

# test passed via Amap API, points ws:116.45911,39.73473 嘉宝花苑 ne:116.48428,39.750821 金茂艺墅
sw_lat, sw_lon = 39.73473, 116.45911
ne_lat, ne_lon = 39.750821, 116.48428
# North-south distance: 1786.58 km
# East-west distance: 2157.66 km

# # Your coordinates
# sw_lat, sw_lon = 39.70152574, 116.43420538
# ne_lat, ne_lon = 39.75155747, 116.52821347

# Calculate north-south distance ( latitude difference )
ns_distance = distVincenty(sw_lat, sw_lon, ne_lat, sw_lon)
print(f"North-south distance: {ns_distance:.2f} m")

# Calculate east-west distance ( longitude difference )
ew_distance = distVincenty(sw_lat, sw_lon, sw_lat, ne_lon)
print(f"East-west distance: {ew_distance:.2f} m")

North-south distance: 1786.58 km
East-west distance: 2157.66 km
